# Churn Intelligence — V2: Processamento Distribuído + Pipeline Assíncrono

Extensão da V1: feature engineering reimplementado em **PySpark** (particionamento, agregações distribuídas) e o streaming processor reimplementado com **asyncio** (múltiplos workers concorrentes). A V1 (pandas + threading) permanece intacta em `src/features/build_features.py` e `src/streaming/event_processor.py`.

## Imports

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import asyncio
import pandas as pd


## Parte 1 — Feature Engineering distribuído (PySpark)

Mesma lógica de negócio da V1 (`build_features.py`), agora usando a DataFrame API do Spark: leitura particionada do CSV, encoding categórico distribuído (dense_rank via window function em vez de `pd.factorize`), imputação de nulos com mediana aproximada por partição e escrita em parquet particionado.

In [ ]:
from src.features.build_features_spark_v2 import build_features_spark, feature_summary, _get_or_create_spark

spark = _get_or_create_spark()

features_df = build_features_spark(
    input_csv="../data/raw/ecommerce_customer_churn_dataset.csv",
    output_dir="../data/processed/features_spark",
    spark=spark,
)

features_df.printSchema()


### Agregação distribuída de exemplo

`groupBy` + agregações via Spark (equivalente distribuído de um `.groupby()` do pandas), útil quando o dataset não cabe mais em um único node.

In [ ]:
feature_summary(features_df).show(20, truncate=False)
spark.stop()


## Parte 2 — Pipeline assíncrono (asyncio, múltiplos workers)

A V1 do streaming (`event_processor.py`) usa uma única thread consumidora. A V2 (`event_processor_async_v2.py`) processa o mesmo contrato de eventos com N corrotinas concorrentes lendo de uma `asyncio.Queue`, cada uma delegando a inferência (bloqueante) para uma thread via `asyncio.to_thread` -- útil quando o processamento de cada evento envolve I/O real (chamada a modelo servido via API, escrita em banco de eventos).

In [ ]:
from src.streaming.event_processor import generate_synthetic_events
from src.streaming.event_processor_async_v2 import AsyncStreamProcessor

events = generate_synthetic_events(n=50)
processor = AsyncStreamProcessor(model_path="../models/rf_model.pkl", n_workers=4)

results = await processor.run(events, delay=0.01)
processor.summary()


### V1 (threading, 1 worker) vs V2 (asyncio, N workers)

Comparação simples de throughput total de processamento entre as duas implementações, sob a mesma carga de eventos sintéticos.

In [ ]:
import time
from src.streaming.event_processor import StreamProcessor

events_v1 = generate_synthetic_events(n=50, seed=42)
t0 = time.perf_counter()
proc_v1 = StreamProcessor(model_path="../models/rf_model.pkl")
proc_v1.run(events_v1, delay=0.01, verbose=False)
t_v1 = time.perf_counter() - t0

events_v2 = generate_synthetic_events(n=50, seed=42)
t0 = time.perf_counter()
proc_v2 = AsyncStreamProcessor(model_path="../models/rf_model.pkl", n_workers=4)
await proc_v2.run(events_v2, delay=0.01, verbose=False)
t_v2 = time.perf_counter() - t0

print(f"V1 - threading (1 worker) : {t_v1:.3f}s")
print(f"V2 - asyncio (4 workers)  : {t_v2:.3f}s")


## Conclusão

A V2 mantém as mesmas regras de negócio da V1 (mesmas features, mesmo mapeamento score→segmento→ação), mudando apenas a camada de execução: processamento distribuído via Spark para feature engineering e concorrência via asyncio para o streaming, preparando o pipeline para volumes maiores e I/O real de produção.